<a href="https://colab.research.google.com/github/bidallei/MIAAD-UACJ/blob/main/Practica_CBR_AlvaroHernandez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

* Asignatura: Sistemas de recomendación
* Profesor: Dra. Karla Olmos Sánchez
* Alumno: **Álvaro Hernández Jarquín**
* Matrícula: 263150
* Fecha de entrega: 2 de noviembre de 2026




# Práctica: Sistema de recomendación basado en casos (CBR) en Python
**Asignatura:** Sistemas de Recomendación · **Nivel:** Maestría  
**Entorno:** Google Colab · **Duración:** 2 horas de clase + 2 horas autónomas

## Objetivo
Implementar un recomendador de cursos de capacitación a partir de experiencias previas, aplicando **recuperar, reutilizar, revisar y retener**. Se utiliza una similitud sencilla para centrar el aprendizaje en el conocimiento representado y la lógica del ciclo CBR.

**Contexto:** Una institución desea recomendar cursos a nuevos participantes según experiencias anteriores. Los datos son ficticios y los resultados no prueban eficacia real.

**Ruta de trabajo:**

1) representar casos;

2) recuperar casos similares;

3) reutilizar una solución;

4) revisar y adaptar;

5) retener una experiencia validada;

6) experimentar y reflexionar.

## Actividad 1. Representar la base de casos
Cada caso describe el **problema** (nivel, área, horas, objetivo), la **solución** (curso) y un **resultado** (satisfacción).

- Nivel: 1 básico, 2 intermedio, 3 avanzado.
- Horas: disponibilidad semanal.
- Satisfacción: escala de 1 a 5.

**Antes de ejecutar:**

¿Qué campos describen al usuario, cuáles la solución y cuáles el resultado?
Los campos que describen al usuario son el nivel, el área, las horas y el objetivo. La solución es el curso, que se va a sugerir al usuario y el resultado es la satisfacción.

¿Qué datos faltan para representar una experiencia real?
Más datos sobre la experiencia del usuario, si ha tomado otros cursos parecidos, el nivel de dificultad, si lo recomendaría o si piensa que está relacionado con sus objetivos.


In [ ]:
import pandas as pd

casos = [
    [1, 1, "Python", 6, "Aprender", "Python básico", 5],
    [2, 2, "Python", 8, "Mejorar", "Python intermedio", 5],
    [3, 3, "Python", 10, "Especializar", "Python avanzado", 4],
    [4, 1, "Datos", 5, "Aprender", "Introducción a datos", 5],
    [5, 2, "Datos", 8, "Mejorar", "Análisis de datos", 4],
    [6, 3, "Datos", 10, "Especializar", "Machine Learning", 5],
    [7, 1, "Seguridad", 6, "Aprender", "Fundamentos de seguridad", 4],
    [8, 2, "Seguridad", 8, "Mejorar", "Seguridad de redes", 5],
    [9, 3, "Seguridad", 12, "Especializar", "Seguridad avanzada", 5],
    [10, 1, "Python", 5, "Aprender", "Python básico", 4],
    [11, 2, "Python", 10, "Mejorar", "Python intermedio", 4],
    [12, 2, "Datos", 6, "Mejorar", "Análisis de datos", 5],
    [13, 1, "Datos", 7, "Aprender", "Introducción a datos", 4],
    [14, 3, "Python", 8, "Especializar", "Python avanzado", 5],
    [15, 2, "Seguridad", 6, "Mejorar", "Seguridad de redes", 4],
    [16, 3, "Datos", 12, "Especializar", "Machine Learning", 4]
]
columnas = ["id", "nivel", "area", "horas", "objetivo", "curso", "satisfaccion"]
df = pd.DataFrame(casos, columns=columnas)
display(df)

,id,nivel,area,horas,objetivo,curso,satisfaccion
0,1,1,Python,6,Aprender,Python básico,5
1,2,2,Python,8,Mejorar,Python intermedio,5
2,3,3,Python,10,Especializar,Python avanzado,4
3,4,1,Datos,5,Aprender,Introducción a datos,5
4,5,2,Datos,8,Mejorar,Análisis de datos,4
5,6,3,Datos,10,Especializar,Machine Learning,5
6,7,1,Seguridad,6,Aprender,Fundamentos de seguridad,4
7,8,2,Seguridad,8,Mejorar,Seguridad de redes,5
8,9,3,Seguridad,12,Especializar,Seguridad avanzada,5
9,10,1,Python,5,Aprender,Python básico,4


**Reflexión:**

¿Por qué la satisfacción no debe confundirse con un atributo del problema?

Porque la satisfacción es el resultado de la recomendación que se dio al usuario.

¿Cuándo una experiencia previa sería poco confiable?

Cuando el usuario no tiene mucha experiencia en un campo. Por ejemplo, en mi experiencia impartiendo clases o entrenamiento, muchas personas piensan que ciertos contenidos, ejercicioso o aprendizajes son muy aburridos o innecesarios, porque no conocen el transcurso total del aprendizaje o proceso.

## Actividad 2. Recuperar experiencias similares
Definimos un nuevo usuario y una similitud ponderada. Los atributos categóricos reciben 1 si coinciden y 0 si difieren; para horas usamos `max(0, 1 - diferencia/12)`.

**Pesos iniciales:** nivel 0.25; área 0.35; horas 0.15; objetivo 0.25. La suma es 1.

**Antes de ejecutar:**

1. ¿Por qué el área pesa más que las horas? ¿Quién debería definir estos pesos?


Porque estamos haciendo una distribución de cursos, ese es el tópico más relevante. Los pesos los debe definir un experto en esta área de enseñanza.

2. ¿Explica la función `max(0, 1 - diferenciar/12)`
Muestra el valor máximo que se obtiene al restar el valor entre 0 y 1 dividido entre 12 que son las horas.

3. ¿Qué representa el 12 y quién debería definirlo?
12 son las horas máximas que se usan en un curso y los definen los datos de los usuarios.

In [ ]:
nuevo_usuario = {"nivel": 2, "area": "Python", "horas": 7, "objetivo": "Mejorar"}
print(nuevo_usuario)

def similitud(usuario, caso):
    sim_nivel = 1 if usuario["nivel"] == caso["nivel"] else 0
    sim_area = 1 if usuario["area"] == caso["area"] else 0
    sim_horas = max(0, 1 - abs(usuario["horas"] - caso["horas"]) / 12)
    sim_objetivo = 1 if usuario["objetivo"] == caso["objetivo"] else 0
    return (0.25 * sim_nivel + 0.35 * sim_area +
            0.15 * sim_horas + 0.25 * sim_objetivo)

df["similitud"] = df.apply(lambda caso: similitud(nuevo_usuario, caso), axis=1)
similares = df.sort_values("similitud", ascending=False).head(3)
display(similares[["id", "nivel", "area", "horas", "objetivo", "curso", "similitud"]])

{'nivel': 2, 'area': 'Python', 'horas': 7, 'objetivo': 'Mejorar'}


,id,nivel,area,horas,objetivo,curso,similitud
1,2,2,Python,8,Mejorar,Python intermedio,0.9875
10,11,2,Python,10,Mejorar,Python intermedio,0.9625
7,8,2,Seguridad,8,Mejorar,Seguridad de redes,0.6375


**Analiza:**

4. ¿Qué tres casos se recuperaron?

Los 3  casos que tienen nivel 2, algunos en área de Python y uno en Seguridad, tienen más de 7 horas y su objetivo es mejorar. La similitud es de 0.63 para arriba.

{'nivel': 2, 'area': 'Python', 'horas': 7, 'objetivo': 'Mejorar'}
| id | nivel | area | horas | objetivo | curso | similitud |
|---|---|---|---|---|---|---|
| 12 | 2 | Python | 8 | Mejorar | Python intermedio | 0.9875 |
| 101 | 12 | Python | 10 | Mejorar | Python intermedio | 0.9625 |
| 78 | 2 | Seguridad | 8 | Mejorar | Seguridad de redes | 0.6375 |

5. ¿Coinciden sus cursos?

2 coinciden y 1 no.

6. ¿Qué cambia al aumentar el peso de `horas` y disminuir el de `area` (manteniendo la suma de pesos en 1)?

Cambian los usuarios y los valores, pero los cursos son los mismos, quizás porque la base de datos es pequeña.

| id | nivel | area | horas | objetivo | curso | similitud |
|---|---|---|---|---|---|---|
| 1 | 2 | Python | 8 | Mejorar | Python intermedio | 0.970833 |
| 10 | 11 | Python | 10 | Mejorar | Python intermedio | 0.912500 |
| 7 | 8 | Seguridad | 8 | Mejorar | Seguridad de redes | 0.820833 |



7. ¿Cuál sería el código equivalente sin utilizar df.apply y lambda?

Se puede lograr así:

```
sim_nivel = (df["nivel"] == nuevo_usuario["nivel"]).astype(int)
sim_area = (df["area"] == nuevo_usuario["area"]).astype(int)
sim_objetivo = (df["objetivo"] == nuevo_usuario["objetivo"]).astype(int)#
```

In [ ]:
# Pregunta 6
pesos2 = {"nivel": 0.25, "area": 0.15, "horas": 0.35, "objetivo": 0.25}

def similitud2(usuario, caso):
    sim_nivel = 1 if usuario["nivel"] == caso["nivel"] else 0
    sim_area = 1 if usuario["area"] == caso["area"] else 0
    sim_horas = max(0, 1 - abs(usuario["horas"] - caso["horas"]) / 12)
    sim_objetivo = 1 if usuario["objetivo"] == caso["objetivo"] else 0

    return (pesos2["nivel"] * sim_nivel +
            pesos2["area"] * sim_area +
            pesos2["horas"] * sim_horas +
            pesos2["objetivo"] * sim_objetivo)

# Usamos una nueva columna 'similitud2' para no sobreescribir la primera
df["similitud2"] = df.apply(lambda caso: similitud2(nuevo_usuario, caso), axis=1)
similares2 = df.sort_values("similitud2", ascending=False).head(3)

print("Resultados con los nuevos pesos (horas=0.35, area=0.15):")
display(similares2[["id", "nivel", "area", "horas", "objetivo", "curso", "similitud2"]])

Resultados con los nuevos pesos (horas=0.35, area=0.15):


,id,nivel,area,horas,objetivo,curso,similitud2
1,2,2,Python,8,Mejorar,Python intermedio,0.970833
10,11,2,Python,10,Mejorar,Python intermedio,0.912500
7,8,2,Seguridad,8,Mejorar,Seguridad de redes,0.820833


## Actividad 3. Reutilizar la solución
En esta versión inicial, se propone el curso del caso más similar. **Similitud no significa automáticamente calidad o viabilidad.**

In [ ]:
mejor_caso = similares.iloc[0]
curso_recomendado = mejor_caso["curso"]
print("Curso recomendado:", curso_recomendado)
print("Caso utilizado:", mejor_caso["id"])
print("Similitud:", round(mejor_caso["similitud"], 3))
print("Satisfacción histórica:", mejor_caso["satisfaccion"])

Curso recomendado: Python intermedio
Caso utilizado: 2
Similitud: 0.987
Satisfacción histórica: 5


**Reflexión:**

8. ¿Qué ocurre si el caso más similar tiene satisfacción 1?

Es un valor muy bajo, sería una mala recomendación, aunque se podría etiquetar como un fracaso.

9. ¿Por qué recuperar y reutilizar son operaciones distintas?

Porque recuperar es cuando usamos el caso de otro usuario que se parece a nuestro nuevo usuario y reutilizar es tomar una decisión respecto a lo que haremos con el caso que recuperamos.

**Ejercicio:**

10. Modifica el criterio para elegir el caso de mayor similitud **entre aquellos con satisfacción histórica de al menos 4**. Explica si esa decisión podría introducir sesgos.

Sí, tiene sesgo porque descarta los que tienen menor calificación de 4, o sea que toma los mejor calificados, los más populares.

In [ ]:
# Casos con satisfacción histórica de al menos 4
casos_exitosos = df[df["satisfaccion"] >= 4]

# Ordenamos por columna de similitud
similares_exitosos = casos_exitosos.sort_values("similitud", ascending=False)

# Tomamos el primero
mejor_caso_exitoso = similares_exitosos.iloc[0]
curso_recomendado_exitoso = mejor_caso_exitoso["curso"]

print("Curso recomendado (solo casos exitosos):", curso_recomendado_exitoso)
print("Caso utilizado:", mejor_caso_exitoso["id"])
print("Similitud:", round(mejor_caso_exitoso["similitud"], 3))
print("Satisfacción histórica:", mejor_caso_exitoso["satisfaccion"])

Curso recomendado (solo casos exitosos): Python intermedio
Caso utilizado: 2
Similitud: 0.987
Satisfacción histórica: 5


## Actividad 4. Revisar y adaptar
Agregamos un catálogo de requisitos mínimos: nivel de entrada y horas semanales. Primero se comprueba la recomendación inicial y luego, si hace falta, se busca una alternativa de la misma área entre las experiencias almacenadas.

**Antes de ejecutar:**

11. ¿Es suficiente que dos usuarios sean similares para recomendarles el mismo curso?

No es suficiente, debemos tomar en cuenta otros datos, aunque sea parecido su perfil.

In [ ]:
catalogo = pd.DataFrame([
    ["Python básico", 1, 5],
    ["Python intermedio", 2, 7],
    ["Python avanzado", 3, 10],
    ["Introducción a datos", 1, 5],
    ["Análisis de datos", 2, 7],
    ["Machine Learning", 3, 10],
    ["Fundamentos de seguridad", 1, 5],
    ["Seguridad de redes", 2, 7],
    ["Seguridad avanzada", 3, 10]
], columns=["curso", "nivel_requerido", "horas_requeridas"])
display(catalogo)

def cumple_requisitos(usuario, curso):
    datos = catalogo[catalogo["curso"] == curso].iloc[0]
    return (usuario["nivel"] >= datos["nivel_requerido"] and
            usuario["horas"] >= datos["horas_requeridas"])

print("¿Cumple requisitos la recomendación inicial?",
      cumple_requisitos(nuevo_usuario, curso_recomendado))

,curso,nivel_requerido,horas_requeridas
0,Python básico,1,5
1,Python intermedio,2,7
2,Python avanzado,3,10
3,Introducción a datos,1,5
4,Análisis de datos,2,7
5,Machine Learning,3,10
6,Fundamentos de seguridad,1,5
7,Seguridad de redes,2,7
8,Seguridad avanzada,3,10


¿Cumple requisitos la recomendación inicial? True


In [ ]:
recomendacion_final = None
for _, caso in df.sort_values("similitud", ascending=False).iterrows():
    curso = caso["curso"]
    if (caso["area"] == nuevo_usuario["area"] and
        cumple_requisitos(nuevo_usuario, curso)):
        recomendacion_final = curso
        break
print("Recomendación final:", recomendacion_final)

Recomendación final: Python intermedio




**Reflexión:**

12 ¿Conviene aplicar restricciones antes o después de recuperar casos? ¿Quién define los requisitos? ¿Qué hacer cuando la respuesta es `None`?

Creo que obtendremos diferentes resultados si las restricciones se hacen antes o después. Si se hace antes se pueden perder algunas posibilidades viables y si se hace después se puede ir adaptando el caso al contexto del usuario. Los requisitos los debe definir alguien que es experto en trayectorias educativas relacionadas con el tópico específico del recomendador. Cuando la respuesta es `None`se pueden cambiar las restricciones o tener una recomendación estándar de inicio en frío.

## Actividad 5. Retener una experiencia
**Supuesto de simulación:** el participante ya tomó el curso y lo evaluó con satisfacción 5. Solo almacenamos una experiencia nueva si hubo recomendación compatible y evaluación favorable. **No debe guardarse una recomendación como experiencia exitosa antes de conocer su resultado.**

In [ ]:
satisfaccion_nueva = 5  # Resultado simulado posterior al curso

if recomendacion_final is not None and satisfaccion_nueva >= 4:
    nuevo_caso = {
        "id": df["id"].max() + 1,
        "nivel": nuevo_usuario["nivel"],
        "area": nuevo_usuario["area"],
        "horas": nuevo_usuario["horas"],
        "objetivo": nuevo_usuario["objetivo"],
        "curso": recomendacion_final,
        "satisfaccion": satisfaccion_nueva
    }
    df = pd.concat([df, pd.DataFrame([nuevo_caso])], ignore_index=True)
    print("Nueva experiencia almacenada")
else:
    print("No se almacenó una experiencia exitosa")

print("Total de casos:", len(df))
display(df.tail(3))

**Reflexión:**

13. ¿Qué problemas surgirían si la base acumula experiencias incorrectas?

Aumentarán las malas recomendaciones porque el sistema se alimenta de su historial.

14. ¿En qué sentido aprende este sistema y en qué difiere de entrenar un modelo de aprendizaje automático?

Cada vez que adquiere una experiencia la acumula y entre más acumula buenas experiencias puede decidir mejor. En el entrenamiento de aprendizaje automático el proceso de entrenamiento empieza y termina. En este sistema aprende hasta dar la nueva recomendación.

**Nota de ejecución:** si vuelves a ejecutar esta celda sin reiniciar el cuaderno, se agregará otra copia de la experiencia simulada. Reinicia y ejecuta desde el principio para repetir el experimento limpiamente.

## Actividad 6. Experimentar con nuevos perfiles
Prueba tres usuarios: A = (nivel 1, Python, 4 horas, Aprender); B = (nivel 3, Datos, 12 horas, Especializar); C = (nivel 2, Seguridad, 5 horas, Mejorar).

In [ ]:
usuarios = [
    {"nivel": 1, "area": "Python", "horas": 4, "objetivo": "Aprender"},
    {"nivel": 3, "area": "Datos", "horas": 12, "objetivo": "Especializar"},
    {"nivel": 2, "area": "Seguridad", "horas": 5, "objetivo": "Mejorar"}
]

for usuario in usuarios:
    puntuaciones = df.apply(lambda caso: similitud(usuario, caso), axis=1)
    ordenados = df.assign(similitud=puntuaciones).sort_values(
        "similitud", ascending=False
    )
    recomendacion = None
    for _, caso in ordenados.iterrows():
        if (caso["area"] == usuario["area"] and
            cumple_requisitos(usuario, caso["curso"])):
            recomendacion = caso["curso"]
            break
    print("\nUsuario:", usuario)
    print("Recomendación:", recomendacion)


Usuario: {'nivel': 1, 'area': 'Python', 'horas': 4, 'objetivo': 'Aprender'}
Recomendación: None

Usuario: {'nivel': 3, 'area': 'Datos', 'horas': 12, 'objetivo': 'Especializar'}
Recomendación: Machine Learning

Usuario: {'nivel': 2, 'area': 'Seguridad', 'horas': 5, 'objetivo': 'Mejorar'}
Recomendación: Fundamentos de seguridad


**Análisis de experimentos:**

¿En qué perfiles existe solución compatible? ¿Por qué no basta con elegir el caso más parecido? Propón una estrategia responsable cuando el sistema no encuentra un curso viable.

PAra el primer usuario recomendó None, así no le recomienda algo para lo que no tiene el mínimo tiempo requerido por el catálogo. En el segundo usuario se sugiere Machine Learning porque tiene tiempo y nivel. Y en el tercero se le sugiere Fundamentos de seguridad, para que esté listo con el nivel necesario, aunque solo tiene 5 horas disponibles.

Una estrategia cuando se recomiende None podría ser avisar al usuario que no hay cursos de 4 horas, pero que hay unos de 5, que si puede hacer el esfuerzo. Y otra forma es que un experto revise el caso para platicar con el usuario debido a que no hay recomendaciones del sistema.

## Actividad integradora: aplicar CBR a su proyecto
***NOTA: SE ENTREGA EN EL COLAB DE MI PROYECTO***


Responde con argumentos y ejemplos de tu dominio:

1. **Representación del conocimiento.** ¿Qué es un caso? Distingue problema, solución y resultado.
2. **Naturaleza del dominio.** ¿Es estructurado o de estructura informal? ¿Qué experiencias serían difíciles de formalizar?
3. **Conocimiento explícito y tácito.** ¿Qué proviene de registros y qué requeriría especialistas o usuarios?
4. **Calidad y ética.** ¿Qué riesgos hay al reutilizar casos sesgados, incorrectos o desactualizados?
5. **Pertinencia.** ¿Por qué utilizarías o descartarías CBR frente a contenido, filtrado colaborativo y reglas?

**Realiza casos similares de tu proyecto y sustituyelos en este código. El resultado es un programa simple de recomendación basado en casos.**


## Entregables
- Cuaderno ejecutado y comentado.
- Respuestas a preguntas previas y posteriores.
- Resultados de los tres perfiles y análisis de fallos.
- Reflexión integradora vinculada a su proyecto.



**Alcance:** el ejercicio usa datos ficticios y reglas de similitud didácticas. Una evaluación empírica real requeriría datos históricos confiables, separación de entrenamiento y prueba, métricas y validación de las soluciones. Además de un apoyo de especialistas del dominio.